In [0]:
## Configure S3 access - credentials stored for later use
# Spark Connect doesn't allow runtime config changes via spark.conf.set()
# Pass credentials directly when reading/writing:
# df = spark.read.option("fs.s3a.access.key", aws_access_key_id) \
#                .option("fs.s3a.secret.key", aws_secret_access_key) \
#                .parquet("s3a://your-bucket/path")

aws_access_key_id = "<aws_access_key_id>"
aws_secret_access_key = "<aws_secret_access_key>"


In [0]:
bucket_name = "healthcare-insurance-bigdata-bipina"

input_path = f"s3a://healthcare-insurance-bigdata-bipina/input-data"
bronze_path = f"s3a://healthcare-insurance-bigdata-bipina/bronze"
silver_path = f"s3a://healthcare-insurance-bigdata-bipina/silver"

In [0]:
s3_options = {
    "fs.s3a.access.key": aws_access_key_id,
    "fs.s3a.secret.key": aws_secret_access_key
}


In [0]:
def read_csv_from_s3(file_path):
    return (
        spark.read
        .options(**s3_options)
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(file_path)
    )

In [0]:
## read csv files from s3 bucket 
patients_df = read_csv_from_s3(f"{input_path}/patients.csv")
subscriber_df = read_csv_from_s3(f"{input_path}/subscriber.csv")
claims_df = read_csv_from_s3(f"{input_path}/claims.csv")
group_subgroup_df = read_csv_from_s3(f"{input_path}/group_subgroup.csv")
policy_groups_df = read_csv_from_s3(f"{input_path}/policy_groups.csv")
hospitals_df = read_csv_from_s3(f"{input_path}/hospitals.csv")
premium_payments_df = read_csv_from_s3(f"{input_path}/premium_payments.csv")

In [0]:
##Validate ingestion
datasets = {
    "patients": patients_df,
    "subscriber": subscriber_df,
    "claims": claims_df,
    "group_subgroup": group_subgroup_df,
    "policy_groups": policy_groups_df,
    "hospitals": hospitals_df,
    "premium_payments": premium_payments_df
}

for name, df in datasets.items():
    print(f"Dataset: {name}")
    print(f"Row count: {df.count()}")
    df.printSchema()
    df.show(5, truncate=False)

Dataset: patients
Row count: 185
root
 |-- patient_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- province: string (nullable = true)
 |-- blood_group: string (nullable = true)
 |-- patient_status: string (nullable = true)

+----------+----------+---------+------+-------------+---+-----------+--------+-----------+--------------+
|patient_id|first_name|last_name|gender|date_of_birth|age|city       |province|blood_group|patient_status|
+----------+----------+---------+------+-------------+---+-----------+--------+-----------+--------------+
|P0001     |Anita     |Rai      |Female|1988-09-07   |37 |Toronto    |ON      |A-         |Active        |
|P0002     |Ramesh    |Poudel   |Male  |2012-09-15   |13 |Toronto    |ON      |A+         |Active        |
|P0003     |Michael

In [0]:
##write raw data to bronze s3
def write_parquet_to_s3(df, output_path):
    (
        df.write
        .options(**s3_options)
        .mode("overwrite")
        .parquet(output_path)
    )

In [0]:
write_parquet_to_s3(patients_df, f"{bronze_path}/patients")
write_parquet_to_s3(subscriber_df, f"{bronze_path}/subscriber")
write_parquet_to_s3(claims_df, f"{bronze_path}/claims")
write_parquet_to_s3(group_subgroup_df, f"{bronze_path}/group_subgroup")
write_parquet_to_s3(policy_groups_df, f"{bronze_path}/policy_groups")
write_parquet_to_s3(hospitals_df, f"{bronze_path}/hospitals")
write_parquet_to_s3(premium_payments_df, f"{bronze_path}/premium_payments")

In [0]:
##Read Bronze data back for transformation
def read_parquet_from_s3(path):
    return (
        spark.read
        .options(**s3_options)
        .parquet(path)
    )

In [0]:
patients_bronze = read_parquet_from_s3(f"{bronze_path}/patients")
subscriber_bronze = read_parquet_from_s3(f"{bronze_path}/subscriber")
claims_bronze = read_parquet_from_s3(f"{bronze_path}/claims")
group_subgroup_bronze = read_parquet_from_s3(f"{bronze_path}/group_subgroup")
policy_groups_bronze = read_parquet_from_s3(f"{bronze_path}/policy_groups")
hospitals_bronze = read_parquet_from_s3(f"{bronze_path}/hospitals")
premium_payments_bronze = read_parquet_from_s3(f"{bronze_path}/premium_payments")

In [0]:
##Check null values
from pyspark.sql import functions as F

def null_count_report(df, dataset_name):
    print(f"Null count report for: {dataset_name}")
    df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]).show(truncate=False)

In [0]:
null_count_report(patients_bronze, "patients_bronze")
null_count_report(subscriber_bronze, "subscriber_bronze")
null_count_report(claims_bronze, "claims_bronze")
null_count_report(group_subgroup_bronze, "group_subgroup_bronze")

Null count report for: patients_bronze
+----------+----------+---------+------+-------------+---+----+--------+-----------+--------------+
|patient_id|first_name|last_name|gender|date_of_birth|age|city|province|blood_group|patient_status|
+----------+----------+---------+------+-------------+---+----+--------+-----------+--------------+
|0         |0         |0        |0     |0            |0  |1   |0       |1          |0             |
+----------+----------+---------+------+-------------+---+----+--------+-----------+--------------+

Null count report for: subscriber_bronze
+-------------+----------+-----------+---------------+-----------------+-------------------+
|subscriber_id|patient_id|subgroup_id|enrollment_date|premium_frequency|subscription_status|
+-------------+----------+-----------+---------------+-----------------+-------------------+
|0            |0         |0          |0              |1                |0                  |
+-------------+----------+-----------+---------

In [0]:
##Check duplicate records
def duplicate_count(df, dataset_name):
    total_count = df.count()
    distinct_count = df.dropDuplicates().count()
    duplicate_records = total_count - distinct_count
    print(f"{dataset_name} duplicate count: {duplicate_records}")

In [0]:
duplicate_count(patients_bronze, "patients")
duplicate_count(subscriber_bronze, "subscriber")
duplicate_count(claims_bronze, "claims")
duplicate_count(group_subgroup_bronze, "group_subgroup")

patients duplicate count: 2
subscriber duplicate count: 2
claims duplicate count: 3
group_subgroup duplicate count: 0


In [0]:
##Clean Patients dataset
patients_silver = (
    patients_bronze
    .dropDuplicates()
    .filter(F.col("patient_id").isNotNull())
    .withColumn("patient_id", F.trim(F.col("patient_id")))
    .withColumn("first_name", F.initcap(F.trim(F.col("first_name"))))
    .withColumn("last_name", F.initcap(F.trim(F.col("last_name"))))
    .withColumn("gender", F.initcap(F.trim(F.col("gender"))))
    .withColumn("date_of_birth", F.to_date(F.col("date_of_birth"), "yyyy-MM-dd"))
    .withColumn("age", F.col("age").cast("int"))
    .withColumn("city", F.coalesce(F.initcap(F.trim(F.col("city"))), F.lit("NA")))
    .withColumn("province", F.coalesce(F.upper(F.trim(F.col("province"))), F.lit("NA")))
    .withColumn("blood_group", F.coalesce(F.upper(F.trim(F.col("blood_group"))), F.lit("NA")))
    .withColumn("patient_status", F.initcap(F.trim(F.col("patient_status"))))
)

In [0]:
##Clean Subscriber dataset
subscriber_silver = (
    subscriber_bronze
    .dropDuplicates()
    .filter(F.col("subscriber_id").isNotNull())
    .withColumn("subscriber_id", F.trim(F.col("subscriber_id")))
    .withColumn("patient_id", F.trim(F.col("patient_id")))
    .withColumn("subgroup_id", F.trim(F.col("subgroup_id")))
    .withColumn("enrollment_date", F.to_date(F.col("enrollment_date"), "yyyy-MM-dd"))
    .withColumn("premium_frequency", F.coalesce(F.initcap(F.trim(F.col("premium_frequency"))), F.lit("NA")))
    .withColumn("subscription_status", F.initcap(F.trim(F.col("subscription_status"))))
)

In [0]:
##Clean Group_subgroup dataset
group_subgroup_silver = (
    group_subgroup_bronze
    .dropDuplicates()
    .filter(F.col("subgroup_id").isNotNull())
    .withColumn("subgroup_id", F.trim(F.col("subgroup_id")))
    .withColumn("group_id", F.trim(F.col("group_id")))
    .withColumn("subgroup_name", F.initcap(F.trim(F.col("subgroup_name"))))
    .withColumn("coverage_description", F.initcap(F.trim(F.col("coverage_description"))))
    .withColumn("annual_premium", F.coalesce(F.col("annual_premium").cast("double"), F.lit(0.0)))
    .withColumn("status", F.initcap(F.trim(F.col("status"))))
)

In [0]:
##Clean Policy Groups, Hospitals, and Premium Payments
policy_groups_silver = (
    policy_groups_bronze
    .dropDuplicates()
    .filter(F.col("group_id").isNotNull())
    .withColumn("group_id", F.trim(F.col("group_id")))
    .withColumn("group_name", F.initcap(F.trim(F.col("group_name"))))
    .withColumn("policy_type", F.initcap(F.trim(F.col("policy_type"))))
    .withColumn("description", F.initcap(F.trim(F.col("description"))))
    .withColumn("profit_margin_rate", F.coalesce(F.col("profit_margin_rate").cast("double"), F.lit(0.0)))
)

hospitals_silver = (
    hospitals_bronze
    .dropDuplicates()
    .filter(F.col("hospital_id").isNotNull())
    .withColumn("hospital_id", F.trim(F.col("hospital_id")))
    .withColumn("hospital_name", F.initcap(F.trim(F.col("hospital_name"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("province", F.upper(F.trim(F.col("province"))))
    .withColumn("hospital_type", F.initcap(F.trim(F.col("hospital_type"))))
)

premium_payments_silver = (
    premium_payments_bronze
    .dropDuplicates()
    .filter(F.col("payment_id").isNotNull())
    .withColumn("payment_id", F.trim(F.col("payment_id")))
    .withColumn("subscriber_id", F.trim(F.col("subscriber_id")))
    .withColumn("payment_date", F.to_date(F.col("payment_date"), "yyyy-MM-dd"))
    .withColumn("premium_amount", F.coalesce(F.col("premium_amount").cast("double"), F.lit(0.0)))
    .withColumn("payment_status", F.initcap(F.trim(F.col("payment_status"))))
)

In [0]:
claims_bronze.show(5, truncate=False)

+--------+-------------+----------+-----------+-------------+----------+-------------+-------------+------------+---------------+--------+
|claim_id|subscriber_id|patient_id|hospital_id|disease_name |claim_date|claim_type   |total_charges|claim_status|approved_amount|severity|
+--------+-------------+----------+-----------+-------------+----------+-------------+-------------+------------+---------------+--------+
|C00001  |S0037        |P0037     |H005       |Asthma       |2025-01-23|Cashless     |35450.0      |Approved    |31074          |Critical|
|C00002  |S0106        |P0106     |H002       |Cancer       |2025-03-07|Cashless     |81391.0      |Pending     |75191          |High    |
|C00003  |S0077        |P0077     |H001       |Diabetes     |2025-05-19|Reimbursement|37715.0      |Rejected    |0              |High    |
|C00004  |S0141        |P0141     |H007       |Heart Disease|2026-01-22|Cashless     |50845.0      |Approved    |41850          |Critical|
|C00005  |S0145        |P01

In [0]:
from pyspark.sql import functions as F

claims_silver = (
    claims_bronze
    .dropDuplicates()
    .filter(F.col("claim_id").isNotNull())
    .withColumn("claim_id", F.trim(F.col("claim_id")))
    .withColumn("subscriber_id", F.trim(F.col("subscriber_id")))
    .withColumn("patient_id", F.trim(F.col("patient_id")))
    .withColumn("hospital_id", F.trim(F.col("hospital_id")))
    .withColumn("disease_name", F.initcap(F.trim(F.col("disease_name"))))
    .withColumn("claim_date", F.to_date(F.col("claim_date"), "yyyy-MM-dd"))
    .withColumn("claim_type", F.coalesce(F.initcap(F.trim(F.col("claim_type"))), F.lit("NA")))
    .withColumn("total_charges", F.coalesce(F.col("total_charges").cast("double"), F.lit(0.0)))
    .withColumn("claim_status", F.initcap(F.trim(F.col("claim_status"))))
    .withColumn("approved_amount", F.coalesce(F.col("approved_amount").cast("double"), F.lit(0.0)))
    .withColumn("severity", F.initcap(F.trim(F.col("severity"))))
)

claims_silver.show(5, truncate=False)

+--------+-------------+----------+-----------+------------+----------+-------------+-------------+------------+---------------+--------+
|claim_id|subscriber_id|patient_id|hospital_id|disease_name|claim_date|claim_type   |total_charges|claim_status|approved_amount|severity|
+--------+-------------+----------+-----------+------------+----------+-------------+-------------+------------+---------------+--------+
|C00007  |S0022        |P0022     |H001       |Cancer      |2025-04-26|Reimbursement|73799.0      |Rejected    |0.0            |Critical|
|C00021  |S0048        |P0048     |H005       |Flu         |2025-11-24|Reimbursement|49984.0      |Pending     |34902.0        |Normal  |
|C00032  |S0080        |P0080     |H001       |Cancer      |2026-05-27|Cashless     |104220.0     |Approved    |71131.0        |Critical|
|C00033  |S0141        |P0141     |H006       |Diabetes    |2026-05-02|Reimbursement|27797.0      |Approved    |25563.0        |Critical|
|C00055  |S0048        |P0048     

In [0]:
##Validate Silver data
silver_datasets = {
    "patients_silver": patients_silver,
    "subscriber_silver": subscriber_silver,
    "claims_silver": claims_silver,
    "group_subgroup_silver": group_subgroup_silver,
    "policy_groups_silver": policy_groups_silver,
    "hospitals_silver": hospitals_silver,
    "premium_payments_silver": premium_payments_silver
}

for name, df in silver_datasets.items():
    print(f"{name} count: {df.count()}")
    df.show(5, truncate=False)

patients_silver count: 183
+----------+----------+---------+------+-------------+---+-----------+--------+-----------+--------------+
|patient_id|first_name|last_name|gender|date_of_birth|age|city       |province|blood_group|patient_status|
+----------+----------+---------+------+-------------+---+-----------+--------+-----------+--------------+
|P0023     |Maya      |Sharma   |Female|1981-01-02   |45 |Brampton   |ON      |B-         |Active        |
|P0024     |Bipana    |Rai      |Female|1943-12-23   |82 |Toronto    |ON      |A+         |Active        |
|P0031     |Kiran     |Brown    |Male  |1996-06-17   |29 |Hamilton   |ON      |B+         |Inactive      |
|P0033     |Rita      |Singh    |Female|2016-03-13   |9  |Scarborough|ON      |A+         |Active        |
|P0042     |Amit      |Davis    |Male  |2010-03-28   |15 |Toronto    |ON      |O+         |Active        |
+----------+----------+---------+------+-------------+---+-----------+--------+-----------+--------------+
only showi

In [0]:
null_count_report(patients_silver, "patients_silver")
null_count_report(subscriber_silver, "subscriber_silver")
null_count_report(claims_silver, "claims_silver")
null_count_report(group_subgroup_silver, "group_subgroup_silver")

Null count report for: patients_silver
+----------+----------+---------+------+-------------+---+----+--------+-----------+--------------+
|patient_id|first_name|last_name|gender|date_of_birth|age|city|province|blood_group|patient_status|
+----------+----------+---------+------+-------------+---+----+--------+-----------+--------------+
|0         |0         |0        |0     |0            |0  |0   |0       |0          |0             |
+----------+----------+---------+------+-------------+---+----+--------+-----------+--------------+

Null count report for: subscriber_silver
+-------------+----------+-----------+---------------+-----------------+-------------------+
|subscriber_id|patient_id|subgroup_id|enrollment_date|premium_frequency|subscription_status|
+-------------+----------+-----------+---------------+-----------------+-------------------+
|0            |0         |0          |0              |0                |0                  |
+-------------+----------+-----------+---------

In [0]:
##Write Silver cleaned data to S3 as CSV
def write_csv_to_s3(df, output_path):
    (
        df.coalesce(1)
        .write
        .options(**s3_options)
        .mode("overwrite")
        .option("header", "true")
        .csv(output_path)
    )

In [0]:
write_csv_to_s3(patients_silver, f"{silver_path}/patients")
write_csv_to_s3(subscriber_silver, f"{silver_path}/subscriber")
write_csv_to_s3(claims_silver, f"{silver_path}/claims")
write_csv_to_s3(group_subgroup_silver, f"{silver_path}/group_subgroup")
write_csv_to_s3(policy_groups_silver, f"{silver_path}/policy_groups")
write_csv_to_s3(hospitals_silver, f"{silver_path}/hospitals")
write_csv_to_s3(premium_payments_silver, f"{silver_path}/premium_payments")